In [ ]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [ ]:
%run ../setup/00_setup_parameters 

In [ ]:
import bronze_schemas
from pyspark.sql import functions as F
import importlib

importlib.reload(bronze_schemas)

def ingest_to_bronze():
    bronze_streame_source_schema = bronze_schemas.get_bronze_schema_for_stream_source()

    # 1. Read stream with Rescue Data enabled
    raw_stream_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiLine", "true")
        .option("rescuedDataColumn", "_rescued_data") # Correct option name without prefix
        .option("cloudFiles.schemaEvolutionMode", "rescue") # Tells Auto Loader to route mismatches there
        .schema(bronze_streame_source_schema)
        .load(landing_volume)
    )

    # 2. Add audit columns
    enriched_df = (
        raw_stream_df
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("insert_timestamp", F.current_timestamp())
    )

    # 3. Check if Auto Loader generated '_rescued_data' in this micro-batch schema
    if "_rescued_data" in enriched_df.columns:
        clean_df = enriched_df.filter(F.col("_rescued_data").isNull() & F.col("trade_id").isNotNull())
        quarantine_df = enriched_df.filter(F.col("_rescued_data").isNotNull() | F.col("trade_id").isNull())
    else:
        # If no rescued data column exists yet, just filter on missing trade_ids
        clean_df = enriched_df.filter(F.col("trade_id").isNotNull())
        quarantine_df = enriched_df.filter(F.col("trade_id").isNull()).withColumn("_rescued_data", F.lit(None))

    bronze_checkpoint = checkpoint_path + "/bronze_streaming_source"
    quarantine_checkpoint = checkpoint_path + "/bronze_quarantine"

    # 4. Write Clean Data to Main Bronze Table
    clean_query = (
        clean_df.drop("_rescued_data")
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", bronze_checkpoint)
        .trigger(availableNow=True)
        .toTable(f"{catalog}.{schema}.bronze_streaming_source")
    )

    # 5. Write Bad Data to Quarantine Table
    quarantine_query = (
        quarantine_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", quarantine_checkpoint)
        .trigger(availableNow=True)
        .toTable(f"{catalog}.{schema}.bronze_quarantined_records")
    )

    clean_query.awaitTermination()
    quarantine_query.awaitTermination()

ingest_to_bronze()